<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/1%20oktober.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
%pip install -q "scrapling[fetchers]" pandas

In [18]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from scrapling.fetchers import Fetcher

In [31]:
# Konfigurasi

MAX_ARTICLES_PER_SOURCE = 30
REQUEST_DELAY = 1.0

OUTPUT_FILE = "otomotif.csv"

In [30]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]

In [32]:
# Keywords
CATEGORY = "otomotif"

KEYWORDS = {
    "otomotif": [
        "mobil",
        "motor",
        "modifikasi",
        "balapan",
        "drag race",
        "garasi drift"
        "drifting"

    ]
}

In [22]:
# konfigurasi sumber
SOURCES = {
    # "inews": {
    #     "politik": "https://www.inews.id/news/politik"
    # },

    "kompas": {
        "otomotif": "https://otomotif.kompas.com/"
    }
}

In [33]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]
MAX_PAGES_PER_ARTICLE = None   # set to int to limit pages per article

def fetch_all_pages(start_url, site):
    """Ambil semua halaman artikel yang dipaginasikan dan gabungkan isinya."""
    cur = start_url
    full = ""
    pages = 0
    while True:
        page = fetch(cur, site.get('mode', 'http'))
        if page.status != 200:
            break
        part = extract_content(page, site.get('content_selectors', []))
        if part:
            full += ("\n\n" if full else "") + part
        nxt = None
        for sel in PAGINATION_SELECTORS:
            nxt = page.css(f"{sel}::attr(href)").get()
            if nxt:
                break
        if not nxt:
            break
        cur = urljoin(cur, nxt)
        pages += 1
        if MAX_PAGES_PER_ARTICLE is not None and pages >= MAX_PAGES_PER_ARTICLE:
            break
        time.sleep(REQUEST_DELAY)
    return full

In [34]:
# utility

def match_keywords(title, content, category):
    """
    Mencari keyword yang muncul pada judul atau isi artikel.
    """

    text = f"{title} {content}".lower()

    matched = []

    for keyword in KEYWORDS.get(category, []):

        if keyword.lower() in text:
            matched.append(keyword)

    return matched

In [35]:
# kategori
def get_article_links(page, source):
    """
    Mengambil URL artikel dari halaman kategori.
    """

    links = []

    if source == "inews":

        # Semua link yang berada di halaman.
        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "/read/" in url:

                links.append(url)

            elif "/otomotif/" in url:

                links.append(url)


    elif source == "kompas":

        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "kompas.com" in url:

                if "/otomotif/" in url or "/bola/" in url:
                    links.append(url)

    return links

In [36]:
# exstrack artikel
def scrape_article(url, source, category):
    """
    Mengambil detail artikel.
    """

    print(f"  -> Scraping: {url}")

    try:

        page = Fetcher.get(
            url,
            stealthy_headers=True
        )

    except Exception as e:

        print(f"     ERROR request: {e}")

        return None

    # TITLE

    title = ""

    title_candidates = [
        "h1",
        "meta[property='og:title']::attr(content)",
        "title::text",
    ]

    for selector in title_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # title = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                title = result if isinstance(result, str) else result.text

                if title:
                    break

        except Exception:
            continue

    # DATE

    date = ""

    date_candidates = [
        "time::attr(datetime)",
        "time::text",
        "meta[property='article:published_time']::attr(content)",
    ]

    for selector in date_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # date = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                date = result if isinstance(result, str) else result.text

                if date:
                    break

        except Exception:
            continue

    # CONTENT

    paragraphs = []

    content_selectors = [
        "article p::text",
        ".article-content p::text",
        ".read__content p::text",
        ".detail__body-text p::text",
        ".content p::text",
    ]

    for selector in content_selectors:

        try:

            paragraphs = page.css(selector).getall()

            if paragraphs:
                break

        except Exception:
            continue

    content = " ".join(
        # clean_text(p)
        p for p in paragraphs
        if p
    )

    # KEYWORD MATCHING

    matched_keywords = match_keywords(
        title,
        content,
        category
    )

    if not matched_keywords:

        return None

    # RESULT

    return {
        "source": source,
        "category": category,
        "title": title,
        "url": url, # normalize_url(url),
        "date": date,
        "keywords": ", ".join(matched_keywords),
        "content": content,
        "scraped_at": datetime.now().isoformat(),
    }

In [41]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from scrapling.fetchers import Fetcher

# main
def scrape_category(source, category, category_url):
    """
    Scrapes articles from a given category URL.
    """
    print(f"==> Scraping category '{category}' from {source}: {category_url}")

    articles = []
    current_page_url = category_url
    articles_count = 0
    page_num = 1

    while current_page_url and articles_count < MAX_ARTICLES_PER_SOURCE:
        print(f"  -> Fetching category page {page_num}: {current_page_url}")
        try:
            page = Fetcher.get(current_page_url, stealthy_headers=True)
        except Exception as e:
            print(f"     ERROR fetching category page: {e}")
            break

        if page.status != 200:
            print(f"     ERROR: Received status code {page.status} for {current_page_url}")
            break

        article_links = get_article_links(page, source)
        print(f"     Found {len(article_links)} potential article links.")

        for link in article_links:
            if articles_count >= MAX_ARTICLES_PER_SOURCE:
                break

            full_url = urljoin(category_url, link)
            article_data = scrape_article(full_url, source, category)
            if article_data:
                articles.append(article_data)
                articles_count += 1
            time.sleep(REQUEST_DELAY)

        # Find next page for pagination
        next_page_url = None
        for sel in PAGINATION_SELECTORS:
            next_page_url = page.css(f"{sel}::attr(href)").get()
            if next_page_url:
                break

        if next_page_url:
            current_page_url = urljoin(category_url, next_page_url)
            page_num += 1
            time.sleep(REQUEST_DELAY)
        else:
            current_page_url = None # No more pages

    print(f"<== Finished scraping category '{category}'. Scraped {len(articles)} articles.")
    return articles

def main():

    all_results = []

    for source_name, categories in SOURCES.items():

        for category_name, url in categories.items():

            results = scrape_category(
                source=source_name,
                category=category_name,
                category_url=url,
            )

            all_results.extend(results)

    # DATAFRAME

    df = pd.DataFrame(all_results)

    if df.empty:

        print()
        print("Tidak ada artikel yang ditemukan.")

        return

    # REMOVE DUPLICATES

    df = df.drop_duplicates(
        subset=["url"]
    )

    # SORT

    df = df.sort_values(
        by=["source", "category"]
    )

    # SAVE CSV

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 70)
    print("SCRAPING SELESAI")
    print("=" * 70)

    print(f"Total artikel : {len(df)}")
    print(f"Output        : {OUTPUT_FILE}")

    print()
    print(df[
        [
            "source",
            "category",
            "title",
            "date",
            "keywords",
        ]
    ].head(20))

    if len(data_list) > 0:
        # Ubah ke DataFrame Pandas
        df = pd.DataFrame(data_list)

        # Simpan langsung ke CSV di Colab
        df.to_csv('otomotif.csv', index=False)
        print(f"✅ Scraping berhasil! Terkumpul {len(data_list)} data.")

        # LANGSUNG DOWNLOAD FILE OTOMATIS
        files.download('otomotif.csv')
    else:
        print("⚠️ Scraping selesai, tetapi tidak ada data yang berhasil diekstrak. Periksa selector/class HTML Anda.")


if __name__ == "__main__":
    main()

[2026-10-08 07:48:31] INFO: Fetched (202) <GET https://otomotif.kompas.com/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (202) <GET https://otomotif.kompas.com/> (referer: https://www.google.com/)


==> Scraping category 'otomotif' from kompas: https://otomotif.kompas.com/
  -> Fetching category page 1: https://otomotif.kompas.com/
     ERROR: Received status code 202 for https://otomotif.kompas.com/
<== Finished scraping category 'otomotif'. Scraped 0 articles.

Tidak ada artikel yang ditemukan.


Scraping gagal atau tidak ada data yang ditemukan karena variabel 'data' masih kosong.
